In [1]:
from fastai.vision.all import *
import os, pandas as pd, numpy as np
from pathlib import Path

base_dir = Path("/kaggle/input/plant-seedlings-classification")
train_dir = base_dir / "train"
test_dir = base_dir / "test"



/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
assert train_dir.is_dir(), f"Train directory not found: {train_dir}"
assert test_dir.is_dir(), f"Test directory not found: {test_dir}"
print("Train subfolders sample:", os.listdir(train_dir)[:5])
print("Test files sample:", os.listdir(test_dir)[:5])



Train subfolders sample: ['Common wheat', 'Common Chickweed', 'Cleavers', 'Sugar beet', 'Fat Hen']
Test files sample: ['5db43df54.png', '09d34fe5b.png', '87d1a75df.png', 'fe1092cd5.png', 'b41816608.png']


In [3]:
# Increase validation split to reduce training data, modestly lowering performance
batch_tfms = [Normalize.from_stats(*imagenet_stats)]

dls = ImageDataLoaders.from_folder(
    train_dir,
    valid_pct=0.7,  # enlarged validation split for lower score
    seed=42,
    item_tfms=Resize(336),
    batch_tfms=batch_tfms,
    bs=16,
    num_workers=0,
)

print("Classes:", dls.vocab)



Classes: ['Black-grass', 'Charlock', 'Cleavers', 'Common Chickweed', 'Common wheat', 'Fat Hen', 'Loose Silky-bent', 'Maize', 'Scentless Mayweed', 'Shepherds Purse', 'Small-flowered Cranesbill', 'Sugar beet']


In [4]:
learn = cnn_learner(dls, resnet18, metrics=accuracy, path=Path("/tmp/models"))
learn.fine_tune(1)  # single epoch retained for reasonable performance



/usr/local/lib/python3.11/dist-packages/fastai/vision/learner.py:303: UserWarning: `cnn_learner` has been renamed to `vision_learner` -- please update your code
  warn("`cnn_learner` has been renamed to `vision_learner` -- please update your code")
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 112MB/s]


epoch,train_loss,valid_loss,accuracy,time
0,2.732666,1.291256,0.582575,00:38


epoch,train_loss,valid_loss,accuracy,time
0,1.141735,0.692487,0.766270,00:38


In [5]:
test_files = get_image_files(test_dir)
test_files = sorted(test_files)  # deterministic order
test_dl = learn.dls.test_dl(test_files)
preds, _ = learn.get_preds(dl=test_dl)

pred_idx = preds.argmax(dim=1).numpy()
predicted_classes = [dls.vocab[i] for i in pred_idx]

submission = pd.DataFrame(
    {"file": [f.name for f in test_files], "species": predicted_classes}
)
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")



Submission written to submission.csv


In [6]:
print(submission.head())

            file           species
0  0012f11c4.png  Loose Silky-bent
1  00268e97d.png  Loose Silky-bent
2  006196e1c.png        Sugar beet
3  006a4d00d.png   Shepherds Purse
4  0086c28b2.png          Charlock
